In [ ]:
#Import our standard libraries.
#Notes: data_profiling is imported to allow us to build a HTML report for EDA overview analysis
#Difflib is imported for fuzzy string matching
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
from data_profiling import ProfileReport


# Load CSV
df = pd.read_csv(r"filename")

# Generate report from data_profiling library
profile = ProfileReport(
    df,
    title="Profiling Report",
    explorative=True
)

profile.to_file("report.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 17/17 [00:00<00:00, 167.12it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Common EDA methods to run on dataframe as a whole
df.shape
df.tail(5)
df.head(5)
df.sample(10, random_state=1)

2114                                 ['Looker', 'Python']
363                   power bi|excel|python|sql|r|tableau
3454             [r, looker, power bi, sql, tableau, sas]
1522                    ['sheets', 'sap', 'excel', 'sql']
1815    ['python', 'spark', 'sql', 'azure', 'snowflake...
Name: job_skills, dtype: str

In [ ]:
#Allows us to catch discrepancies with what we expect from the data vs reality. We expect job_work_from_home to have T/F (2 values), but we see 17 unique values instead.
df.nunique()

job_title_short            63
Job Title                 192
job_location              166
JOB_VIA                    39
job schedule type          50
job_work_from_home         17
search_location            53
Job Posted Date          4500
job_no_degree_mention      17
job health insurance       17
Country                    53
salary_rate                16
Salary Year Avg           628
salary_hour_avg           108
company name              277
job_skills               4550
Job_Type_Skills          4421
dtype: int64

In [ ]:
#Common EDA methods to run on columns for further data inspection
df.columns
list(df.columns)
df.info()
df.describe()                    # numeric columns by default
df.describe(include='all')       # every column
df['job_skills'].sample(10)

,job_title_short,Job Title,job_location,JOB_VIA,job schedule type,job_work_from_home,search_location,Job Posted Date,job_no_degree_mention,job health insurance,Country,salary_rate,Salary Year Avg,salary_hour_avg,company name,job_skills,Job_Type_Skills
count,5226,5225,5225,5225,5162,5225,5225,5225,5225,5225,5225,1874,1791,1049,5159,4976,4986
unique,63,192,166,39,50,17,53,4500,17,17,53,16,628,108,277,4550,4421
top,Data Analyst,Data Analyst,Anywhere,via LinkedIn,Full-time,False,United States,TBD,False,False,United States,year,-,-,Meridian Pharma,[],[]
freq,1085,636,621,1623,3380,1913,2723,14,1418,1883,2710,540,393,473,127,59,69


In [ ]:
#Allows us to see columns with missing data, and can break down missing daats % per column
(df.isna().mean() * 100).round(1)        # percent missing per column
df.isnull().sum()

job_title_short             0
Job Title                   1
job_location                1
JOB_VIA                     1
job schedule type          64
job_work_from_home          1
search_location             1
Job Posted Date             1
job_no_degree_mention       1
job health insurance        1
Country                     1
salary_rate              3352
Salary Year Avg          3435
salary_hour_avg          4177
company name               67
job_skills                250
Job_Type_Skills           240
dtype: int64

In [ ]:
#Allows us to perform some initial EDA on job title column, where we can see multiple listings for the same role, as well as any extra spaces that will need to be cleaned up.
sorted(df['Job Title'].dropna().unique().tolist())

[' Business Analyst',
 ' Business Analyst (Hybrid)',
 ' Business Analyst - Analytics',
 ' Business Analyst - Remote',
 ' Cloud Engineer',
 ' Cloud Engineer - Analytics',
 ' Cloud Engineer II',
 ' Data Analyst',
 ' Data Analyst (Hybrid)',
 ' Data Analyst - Analytics',
 ' Data Analyst - Remote',
 ' Data Analyst II',
 ' Data Engineer',
 ' Data Engineer - Analytics',
 ' Data Engineer - Remote',
 ' Data Scientist',
 ' Data Scientist (Hybrid)',
 ' Data Scientist - Remote',
 ' Data Scientist II',
 ' Machine Learning Engineer (Hybrid)',
 ' Machine Learning Engineer - Analytics',
 ' Machine Learning Engineer II',
 ' Senior Data Analyst',
 ' Senior Data Analyst (Hybrid)',
 ' Senior Data Analyst - Analytics',
 ' Senior Data Analyst - Remote',
 ' Senior Data Engineer',
 ' Senior Data Engineer (Hybrid)',
 ' Senior Data Engineer - Analytics',
 ' Senior Data Engineer - Remote',
 ' Senior Data Engineer II',
 ' Senior Data Scientist',
 ' Senior Data Scientist (Hybrid)',
 ' Senior Data Scientist - Analy

In [ ]:
#Created function to go through dataset and track which columns have number of whitespaces to clean up, as well show values that were merged because they differed only by capitalization and/or extra spaces.

def whitespace_case_report(df):
    rows = []
    for col in df.select_dtypes(include=['object', 'string']).columns:
        s = df[col].dropna().astype(str)
        stripped = s.str.strip()
        normalized = stripped.str.lower()
        rows.append({
            'column': col,
            'non_null': len(s),
            'cells_with_edge_spaces': (s != stripped).sum(),
            'nunique_before': s.nunique(),
            'nunique_after': normalized.nunique(),
        })
    out = pd.DataFrame(rows)
    out['values_merged'] = out['nunique_before'] - out['nunique_after']
    return out.sort_values('values_merged', ascending=False).reset_index(drop=True)

whitespace_case_report(df)

,column,non_null,cells_with_edge_spaces,nunique_before,nunique_after,values_merged
0,company name,5159,263,277,115,162
1,Job Title,5225,242,192,66,126
2,job_location,5225,194,166,49,117
3,job_title_short,5226,318,63,22,41
4,job_skills,4976,317,4550,4515,35
5,search_location,5225,151,53,26,27
6,Country,5225,145,53,26,27
7,JOB_VIA,5225,129,39,13,26
8,job schedule type,5162,150,50,28,22
9,job_work_from_home,5225,0,17,11,6


In [ ]:
#Created Lambda function to go through dataframe and check for NaN/Unknown data and return sum per column for later cleanup
placeholders = {'n/a', 'na', 'none', 'null', '-', '?', 'unknown', ''}
txt = df.select_dtypes(include=['object', 'string'])
txt.apply(lambda c: c.dropna().astype(str).str.strip().str.lower().isin(placeholders).sum())


job_title_short            0
Job Title                  0
job_location               0
JOB_VIA                    0
job schedule type         15
job_work_from_home         0
search_location            0
Job Posted Date            0
job_no_degree_mention      0
job health insurance       0
Country                    0
salary_rate              350
Salary Year Avg          393
salary_hour_avg          473
company name               0
job_skills                 0
Job_Type_Skills            0
dtype: int64

In [ ]:
#df['Country'].value_counts()
#df['job_work_from_home'].value_counts(dropna=False)
#df['JOB_VIA'].value_counts().sort_index()
#

[' via Built In',
 ' via Company Website',
 ' via Glassdoor',
 ' via Indeed',
 ' via LinkedIn',
 ' via ZipRecruiter']

In [ ]:
#Quick EDA method on Job Posted Date that tells us length of data showing inconsistencies
df['Job Posted Date'].astype(str).str.len().value_counts()

Job Posted Date
19.0    2886
16.0     654
10.0     557
12.0     409
20.0     336
9.0      268
11.0     100
3.0       14
15.0       1
Name: count, dtype: int64

In [ ]:
#EDA on Salary. Salary can have many possible entries (-, USD, K, $, .xK). Code below converts non empty values to numbers if possible, then finds entries that were unable to convert to track issue data down.
nums = pd.to_numeric(df['Salary Year Avg'], errors='coerce')

df['Salary Year Avg'].notna().sum()
nums.notna().sum()  

failed = df['Salary Year Avg'][nums.isna() & df['Salary Year Avg'].notna()]
failed.value_counts().head(10)

np.int64(676)

In [ ]:
#More simple salary EDA
salary = df['Salary Year Avg']

salary.describe()
salary.sort_values().head(10)      # smallest 10
salary.sort_values().tail(10)      # largest 10


count     1791
unique     628
top          -
freq       393
Name: Salary Year Avg, dtype: object

In [ ]:
#EDA to find duplicates in the data, both count and rturned as a df
df.duplicated().sum()  
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10)

,job_title_short,Job Title,job_location,JOB_VIA,job schedule type,job_work_from_home,search_location,Job Posted Date,job_no_degree_mention,job health insurance,Country,salary_rate,Salary Year Avg,salary_hour_avg,company name,job_skills,Job_Type_Skills
729,Data Analyst,Data Analyst (Hybrid),"Berlin, Germany",via LinkedIn,Contractor,FALSE,Germany,2023-10-07,True,False,DE,NaN,NaN,NaN,Highland Retail,"['LOOKER', 'SQL SERVER', 'SQL']","{'analyst_tools': ['looker'], 'databases': ['s..."
1565,Data Analyst,Data Analyst (Hybrid),"Berlin, Germany",via LinkedIn,Contractor,FALSE,Germany,2023-10-07,True,False,DE,NaN,NaN,NaN,Highland Retail,"['LOOKER', 'SQL SERVER', 'SQL']","{'analyst_tools': ['looker'], 'databases': ['s..."
2625,Data Engineer,Data Engineer - Remote,Anywhere,LinkedIn,Full-time and Part-time,yes,Germany,02/26/2023 21:29,1,False,DE,NaN,NaN,NaN,Juniper Cloud Inc.,"['python', 'spark', 'sql', 'kafka', 'snowflake...",programming:python|sql|scala;libraries:spark|k...
2706,Data Engineer,Data Engineer - Remote,Anywhere,LinkedIn,Full-time and Part-time,yes,Germany,02/26/2023 21:29,1,False,DE,NaN,NaN,NaN,Juniper Cloud Inc.,"['python', 'spark', 'sql', 'kafka', 'snowflake...",programming:python|sql|scala;libraries:spark|k...
611,Data Scientist,Data Scientist,"Hyderabad, India",via LinkedIn,Part-time,False,India,2023-09-08,false,True,India,NaN,NaN,Not specified,Pinecrest Education,"['r', 'tensorflow', 'tableau']","{'programming': ['r'], 'libraries': ['tensorfl..."
4316,Data Scientist,Data Scientist,"Hyderabad, India",via LinkedIn,Part-time,False,India,2023-09-08,false,True,India,NaN,NaN,Not specified,Pinecrest Education,"['r', 'tensorflow', 'tableau']","{'programming': ['r'], 'libraries': ['tensorfl..."
2648,Data Scientist,Data Scientist,"Vancouver, Canada",LinkedIn,Full-time,false,Canada,2023-06-22 17:30:19,Yes,N,Canada,NaN,Not specified,NaN,NaN,"r , spark , pandas , tableau , r","{'programming': ['r'], 'analyst_tools': ['tabl..."
2659,Data Scientist,Data Scientist,"Vancouver, Canada",LinkedIn,Full-time,false,Canada,2023-06-22 17:30:19,Yes,N,Canada,NaN,Not specified,NaN,NaN,"r , spark , pandas , tableau , r","{'programming': ['r'], 'analyst_tools': ['tabl..."
700,Business Analyst,Business Analyst,"Kansas City, MO",via Indeed,Full-time,0,United States,2023-04-20 01:43:09,N,true,United States,NaN,NaN,NaN,Vantage Consulting,"[tableau, sql, sap, sheets, excel, powerpoint,...","{'analyst_tools': ['tableau', 'sheets', 'excel..."
793,Business Analyst,Business Analyst,"Kansas City, MO",via Indeed,Full-time,0,United States,2023-04-20 01:43:09,N,true,United States,NaN,NaN,NaN,Vantage Consulting,"[tableau, sql, sap, sheets, excel, powerpoint,...","{'analyst_tools': ['tableau', 'sheets', 'excel..."


In [ ]:
#Lastly is a comprehensive function that for each column prints: the dtype, missing count, number of unique values, how many cells have padding, how many unique values remain after normalizing case/space, whether there are line breaks or non-ASCII characters, text length range, and either the full value list (for small categories) or the top values plus a sample.
def profile(df, max_unique=25):
    for c in df.columns:
        s = df[c]
        print('=' * 60)
        print(repr(c), '| dtype:', s.dtype)
        print('missing:', s.isna().sum(), f'({s.isna().mean():.1%})', '| unique:', s.nunique())

        st = s.dropna().astype(str)
        print('padded with spaces:', (st != st.str.strip()).sum(),
              '| unique after strip+lower:', st.str.strip().str.lower().nunique(),
              '| has newline:', st.str.contains('\n').sum(),
              '| non-ASCII:', st.str.contains(r'[^\x00-\x7F]').sum())
        print('text length min/max:', st.str.len().min(), st.str.len().max())

        if s.nunique() <= max_unique:
            print(s.value_counts(dropna=False).to_string())
        else:
            print('top 8:', s.value_counts().head(8).to_dict())
            print('random sample:', s.dropna().sample(3, random_state=0).tolist())

profile(df)

'job_title_short' | dtype: object
missing: 0 (0.0%) | unique: 63
padded with spaces: 318 | unique after strip+lower: 22 | has newline: 0 | non-ASCII: 0
text length min/max: 12 53
top 8: {'Data Analyst': 1085, 'Data Engineer': 847, 'Data Scientist': 763, 'Business Analyst': 315, 'Senior Data Engineer': 293, 'Senior Data Scientist': 282, 'Senior Data Analyst': 238, 'Software Engineer': 208}
random sample: ['DATA SCIENTIST', 'Data Engineer', 'Data Scientist']
'Job Title' | dtype: object
missing: 1 (0.0%) | unique: 192
padded with spaces: 242 | unique after strip+lower: 66 | has newline: 0 | non-ASCII: 0
text length min/max: 9 38
top 8: {'Data Analyst': 636, 'Data Engineer': 479, 'Data Scientist': 416, 'Business Analyst': 185, 'Data Analyst - Remote': 159, 'Data Analyst - Analytics': 150, 'Data Analyst (Hybrid)': 150, 'Data Analyst II': 144}
random sample: ['Data Scientist', 'Data Scientist', 'Data Scientist - Remote']
'job_location ' | dtype: object
missing: 1 (0.0%) | unique: 166
padded 